In [1]:
import os

In [2]:
%pwd

'p:\\OM\\Projects\\End-to-End-ML-\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'p:\\OM\\Projects\\End-to-End-ML-'

In [11]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen = True)
class ModelTrainerConfig:
    root_dir : Path
    train_data_path : Path
    test_data_path : Path
    model_name : str
    alpha : float
    l1_ratio : float
    TARGET_COLUMN : str


In [6]:
from src.wine.constants import *
from src.wine.utils.common import read_yaml, create_directories

In [12]:
class ConfigurationManager:
    def __init__(
        self,
        config_file_path = CONFIG_FILE_PATH,
        params_file_path = PARAMS_FILE_PATH,
        schema_file_path = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig : 
        config = self.config.model_trainer
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir = config.root_dir,
            train_data_path = config.train_data_path,
            test_data_path = config.test_data_path,
            model_name = config.model_name,
            alpha = params.alpha,
            l1_ratio = params.l1_ratio,
            TARGET_COLUMN = schema.name
        )

        return model_trainer_config

In [8]:
import os
from src.wine.logging import logger
from sklearn.linear_model import ElasticNet
import pandas as pd
import joblib

In [13]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):
        train_data = pd.read_csv(self.config.train_data_path)
        test_data = pd.read_csv(self.config.test_data_path)

        x_train = train_data.drop([self.config.TARGET_COLUMN], axis = 1)
        x_test = test_data.drop([self.config.TARGET_COLUMN], axis=1)
        y_train = train_data[[self.config.TARGET_COLUMN]]
        y_test = test_data[[self.config.TARGET_COLUMN]]

        lr = ElasticNet(alpha = self.config.alpha, l1_ratio = self.config.l1_ratio, random_state = 42)
        lr.fit(x_train, y_train)

        joblib.dump(lr, os.path.join(self.config.root_dir, self.config.model_name))

In [14]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer_config = ModelTrainer(config=model_trainer_config)
    model_trainer_config.train()
except Exception as e:
    raise e

[ 2026-06-22 14:16:45,211 : INFO : common : yaml file: config\config.yaml loaded successfully ]
[ 2026-06-22 14:16:45,214 : INFO : common : yaml file: params.yaml loaded successfully ]
[ 2026-06-22 14:16:45,218 : INFO : common : yaml file: schema.yaml loaded successfully ]
[ 2026-06-22 14:16:45,220 : INFO : common : created directory at: artifacts ]
[ 2026-06-22 14:16:45,223 : INFO : common : created directory at: artifacts/model_trainer ]
